# 02 - Synthetic data generation

## Objective

The objective of this notebook is to design and implement a synthetic data generator for industrial electrical coils.

The generator will simulate a simplified manufacturing process based on:

- product characteristics;
- process parameters;
- environmental conditions;
- physical and process assumptions defined in the domain analysis.

Each generated observation represents one manufactured coil.

The synthetic dataset will later be used for:

- exploratory data analysis;
- predictive modelling;
- anomaly detection;
- explainability experiments.

## Coil production representation

```text
Product characteristics
       │
       ▼
Process parameters ───────────┐
                              │
Environmental conditions ─────┤
                              ▼
                     Manufacturing model
                              │
                              ▼
                   Process quality indicators
                              │
              ┌───────────────┼────────────────┬───────────────┐
              ▼               ▼                ▼               ▼
         Resistance        Rigidity    Insulation resistance  Displacement
              │               │                │               │
              └───────────────┴────────────────┴───────────────┘
                              │
                              ▼
                        PASS / FAIL

In more detail:

Product characteristics
│
├── Product family
├── Coil geometry
├── Wire cross-section
└── Nominal number of turns

Process quality indicators
│
├── Winding quality
├── Winding regularity
├── Mechanical stress
└── Welding quality
```

## Variables generation strategy and schema

### Generated variables

These variables are directly generated by the synthetic data generator.

### Calculated variables

These variables are calculated from physical relationships or process assumptions.

### Derived variables

These variables are derived from calculated measurements and quality criteria.

| Category   | Variables                                                              |
| ---------- | ---------------------------------------------------------------------- |
| Generated  | Number of turns, speed, tension, welding temperature, welding duration |
| Generated  | Ambient temperature, humidity                                          |
| Generated  | Product characteristics                                                |
| Calculated | Wire length                                                            |
| Calculated | Process quality indicators                                             |
| Calculated | Resistance, rigidity, insulation resistance, displacement              |
| Derived    | PASS / FAIL                                                            |

## Electric displacement: definition and dimensional analysis

The electric displacement field is defined as:

$$
\mathbf{D} = \varepsilon_0\mathbf{E} + \mathbf{P}
$$

where:

- $\mathbf{D}$ = electric displacement field
- $\varepsilon_0$ = vacuum permittivity
- $\mathbf{E}$ = electric field
- $\mathbf{P}$ = electric polarization

For a linear dielectric material:

$$
\mathbf{D} = \varepsilon\mathbf{E}
$$

with:

$$
\varepsilon = \varepsilon_r\varepsilon_0
$$

In more detail:

$$
\mathbf{P} = \varepsilon_0\chi_e\mathbf{E}
$$

where:

$$
\chi_e = \text{electric susceptibility (dimensionless)}
$$

$$
\chi_e = \varepsilon_r - 1
$$

$$
\begin{aligned}
\mathbf{D} &= \varepsilon_0\mathbf{E} + \mathbf{P} \\
            &= \varepsilon_0\mathbf{E} + \varepsilon_0\chi_e\mathbf{E} \\
            &= \varepsilon_0(1+\chi_e)\mathbf{E} \\
            &= \varepsilon_0\varepsilon_r\mathbf{E} \\
            &= \varepsilon\mathbf{E}
\end{aligned}
$$

### Dimensional analysis: expression in fundamental SI units

The electric displacement field has units:
$$
[D] = \frac{C}{m^2}
$$

Since:

$$
C = A \cdot s
$$

we obtain:

$$
[D] = A\,s\,m^{-2}
$$

For the vacuum permittivity:

$$
[N] = kgms^{-2}
$$

Therefore:

$$
\begin{aligned}
\left[\varepsilon_0\right] &= \frac{C^2}{N\,m^2} \\
                           &= \frac{(A\,s)^2}{(kg\,m\,s^{-2})m^2} \\
                           &= kg^{-1}m^{-3}s^4A^2
\end{aligned}
$$

## Resistivity curves for different RRR values

<p align="center">
  <img src="../data/synthetic/resistivity_RRR.png" width="700">
</p>

## Resistivity: mathematical modelling

We can write the resistivity in the following way:

$$
\rho(T) = \rho_i(T) + \rho_0
$$

where:

- $\rho_i(T)$: intrinsic component (ideal), which depends on temperature;
- $\rho_0$: residual component, which is approximately constant and depends on the material quality;
- $RRR$: Residual Resistivity Ratio, which can be used as an indicator of material quality and to estimate the magnitude of $\rho_0$.

From the graph we have:

$$
RRR = \frac{\rho(273\,K)}{\rho(4\,K)}
$$

We can approximate:

$$
\rho_0 \approx \rho(4\,K)
$$

Therefore:

$$
\begin{aligned}
RRR
&= \frac{\rho(273\,K)}{\rho(4\,K)} \\
&\approx \frac{\rho(273\,K)}{\rho_0}
\end{aligned}
$$

and therefore:

$$
\boxed{
\rho_0 \approx \frac{\rho(273\,K)}{RRR}
}
$$

We can think of $RRR$ as a quality indicator:

$$
RRR \uparrow
\quad\Longrightarrow\quad
\rho_0 \downarrow
\quad\Longrightarrow\quad
\text{higher material purity}
$$

### First-order Taylor approximation of resistivity

Now we can write the first-order Taylor expansion for the resistivity, with the Peano remainder, centered at:

$$
T_0 = 273.15\,K
$$

We start from:

$$
\rho(T)=\rho_i(T)+\rho_0
$$

Therefore:

$$
\begin{aligned}
\rho(T)
&=
\rho_0+\rho_i(T_0)
+
\left.
\frac{d\rho_i}{dT}
\right|_{T_0}
(T-T_0)
+
o(T-T_0)
\end{aligned}
$$

From the graph we have:

$$
T_0=273.15\,K=0^\circ C
$$

and:

$$
\rho_i(0^\circ C)
=
1.545\times10^{-8}\,\Omega\cdot m
$$

as well as:

$$
\left.
\frac{d\rho_i}{dT}
\right|_{T_0}
=
6.7\times10^{-11}\,\Omega\cdot m/K
$$

Therefore, for the temperature $T_K$ in Kelvin:

$$
\rho(T_K)
=
\rho_0
+
1.545\times10^{-8}
+
6.7\times10^{-11}(T_K-273.15)
+
o(T_K-273.15)
$$

We know that:

$$
T_K=T_C+273.15
$$

therefore:

$$
T_C=T_K-273.15
$$

Finally, we obtain the first-order approximation in terms of Celsius temperature:

$$
\boxed{
\rho(T_C)
\approx
\rho_0
+
1.545\times10^{-8}
+
6.7\times10^{-11}T_C
}
$$

where $T_C$ is expressed in $^\circ C$.

### First-order approximation of $\Delta\rho$

We have that:

$$
\rho(T_0+dT_K)-\rho_i(T_0)
=
\rho_0
+
\rho_i'(T_0)dT_K
+
o(dT_K)
$$

as

$$
dT_K \to 0\,K.
$$

Synthetically:

$$
\Delta\rho(T_K)
=
\rho_0
+
\rho_i'(T_0)dT_K
+
o(dT_K),
\qquad
dT_K \to 0\,K.
$$

In conclusion:

$$
\Delta\rho(273.15\,K)
\approx
\rho_0
+
d\rho_i(273.15\,K)
$$

in a neighbourhood of $273.15\,K$.

Equivalently, in Celsius:

$$
\Delta\rho(0^\circ C)
\approx
\rho_0
+
d\rho_i(0^\circ C)
$$

in a neighbourhood of $0^\circ C$.